# Session 2 · NLP Applied to Emotion Detection
## Activity 2 — Scaling Up: pandas Preprocessing Pipeline · **SOLUTION**

In [ ]:
import pandas as pd
from collections import Counter

EMOTICONS = {":)", ":(", ":D", ":P", ";)", ":/", ":o", "XD", "<3",
             ":-)", ":-(", ":-D", ":-P"}

STOPWORDS = {
    "a", "an", "the", "in", "on", "at", "to", "for", "of", "with", "by", "from",
    "and", "but", "or", "so",
    "is", "are", "was", "were", "be", "been", "being", "do", "does", "did",
    "have", "has", "had", "will", "would", "could", "should",
    "i", "you", "he", "she", "it", "we", "they",
    "me", "him", "her", "us", "them",
    "my", "your", "his", "its", "our", "their",
    "this", "that", "these", "those", "just", "also", "very",
}

def tokenise(text, keep_emoticons=True):
    tokens, buffer, i = [], "", 0
    while i < len(text):
        if keep_emoticons:
            two_chars = text[i:i+2]
            if two_chars in EMOTICONS:
                if buffer.strip(): tokens.append(buffer.strip()); buffer = ""
                tokens.append(two_chars); i += 2; continue
        char = text[i]
        if char == " ":
            if buffer.strip(): tokens.append(buffer.strip()); buffer = ""
        elif char.isalpha() or char.isdigit():
            buffer += char
        i += 1
    if buffer.strip(): tokens.append(buffer.strip())
    return tokens

def pipeline(text, keep_emoticons=True, apply_stopwords=True, min_length=2):
    text = text.lower()
    tokens = tokenise(text, keep_emoticons=keep_emoticons)
    if apply_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS]
    tokens = [t for t in tokens if len(t) >= min_length or t in EMOTICONS]
    return tokens

## Part B — Load the dataset

In [ ]:
# TODO 1 — load
df = pd.read_csv("reviews.csv")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

In [ ]:
# TODO 2a — first rows
df.head()

In [ ]:
# TODO 2b — distribution + missing values
print("Label distribution:")
print(df["sentiment"].value_counts())
print()
print("Missing values:")
print(df.isnull().sum())

## Part C — Apply the pipeline

In [ ]:
# TODO 3 — apply pipeline
df["tokens"] = df["text"].apply(pipeline)

In [ ]:
# TODO 4 — inspect first 5 rows
for idx, row in df.head(5).iterrows():
    print(f"[{row['sentiment']:8s}] {row['text'][:55]}...")
    print(f"           → {row['tokens']}")
    print()

## Part D — Analyse the results

In [ ]:
# TODO 5 — token count statistics
df["n_tokens"] = df["tokens"].apply(len)
print(f"Mean tokens per review : {df['n_tokens'].mean():.1f}")
print(f"Min tokens             : {df['n_tokens'].min()}")
print(f"Max tokens             : {df['n_tokens'].max()}")

In [ ]:
# TODO 6 — longest review
idx_max = df["n_tokens"].idxmax()
print("Longest review (after preprocessing):")
print(f"  Text  : {df.loc[idx_max, 'text']}")
print(f"  Tokens: {df.loc[idx_max, 'tokens']}")

In [ ]:
# TODO 7 — top 20 tokens overall
all_tokens = [token for token_list in df["tokens"] for token in token_list]
top_20 = Counter(all_tokens).most_common(20)

print("Top 20 most frequent tokens:")
for token, count in top_20:
    print(f"  {token:20s} {count}")

## Part E — Positive vs negative vocabulary

In [ ]:
# TODO 8 — compare vocabularies
pos_tokens = [t for tl, s in zip(df["tokens"], df["sentiment"]) for t in tl if s == "positive"]
neg_tokens = [t for tl, s in zip(df["tokens"], df["sentiment"]) for t in tl if s == "negative"]

pos_freq = Counter(pos_tokens)
neg_freq = Counter(neg_tokens)

print("Most common in POSITIVE reviews:")
for token, count in pos_freq.most_common(10):
    print(f"  {token:20s} {count}")
print()
print("Most common in NEGATIVE reviews:")
for token, count in neg_freq.most_common(10):
    print(f"  {token:20s} {count}")

## Part F — Answers

**Q1.** Some neutral tokens (like `'not'`, `'product'`) may appear in both lists. These are weak discriminators — a classifier relying solely on frequency would struggle with them. This motivates TF-IDF (Session 3): it *down-weights* tokens that appear equally across classes.

**Q2.** Emoticons like `:)` and `:(` may appear in the top positive/negative lists respectively — precisely because they carry strong sentiment. Removing them (`keep_emoticons=False`) would reduce discriminative signal, especially for informal or social-media text.

**Q3.** Words like `'product'`, `'experience'` or `'would'` might appear frequently in both classes. Adding them to `STOPWORDS` would reduce noise — but you must verify they don't appear in critical expressions like `'would not recommend'`. Always check in context before removing.

**Q4.** The next step (Session 3) is converting token lists to **numeric vectors**: first Bag of Words (count how many times each vocabulary word appears), then TF-IDF (weight by how discriminative each word is). These vectors are the actual input to machine learning models.

In [ ]:
print("We now have clean token lists like:")
print(f"  {df.loc[0, 'tokens']}")
print()
print("Session 3: how do we turn this list into NUMBERS?")
print("→ Bag of Words  →  TF-IDF  →  Numeric representations")